# 02 - Chunking Strategies Comparison

**Phase 1, Step 2** of the Ingestion & Chunking Lifecycle.

Compares 4 chunking strategies across all document formats:

| Strategy | Approach | Key Parameter |
|---|---|---|
| **Fixed** | Character-based split with overlap | `chunk_size`, `chunk_overlap` |
| **Structural** | Split on logical separators (headers, sections) | Custom separator list |
| **Semantic** | Group by embedding similarity | Similarity threshold |
| **Custom RBAC** | Isolate sensitive fragments (PII/confidential) into independent chunks | Regex patterns + sensitivity rules |

### Metrics collected for Phase 2 analysis:
- Number of chunks produced
- Avg/min/max chunk size (characters)
- Boundary coherence (% of chunks ending on sentence boundary)
- Context preservation (overlap ratio)
- Sensitive content isolation (% of PII confined to dedicated chunks)

In [ ]:
import json
import re
import statistics
from typing import Optional

import numpy as np
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter

# Load documents exported from notebook 01
with open("../../../data/results/notebook_results/loaded_documents.json", "r", encoding="utf-8") as f:
    raw = json.load(f)

all_documents: dict[str, list[Document]] = {}
for filename, doc_list in raw.items():
    all_documents[filename] = [
        Document(page_content=d["page_content"], metadata=d["metadata"])
        for d in doc_list
    ]

print(f"Loaded {len(all_documents)} documents from cache")
for name, docs in all_documents.items():
    total_chars = sum(len(d.page_content) for d in docs)
    print(f"  {name}: {total_chars} chars across {len(docs)} section(s)")

## Metrics Framework

Unified metrics function applied to every strategy's output.

In [2]:
# Regex patterns for detecting sensitive content (PII and confidential data)
PII_PATTERNS = {
    "email": r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
    "iban": r"[A-Z]{2}\d{2}[\s]?[A-Z0-9]{4}[\s]?[\d]{4}[\s]?[\d]{4}[\s]?[\d]{4}[\s]?[\d]{4}[\s]?[\d]{0,3}",
    "ip_address": r"\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b",
    "password": r"(?i)(?:password|contrase[ñn]a|pwd|override[_ ]?password)\s*[:=]\s*['\"]?([^\s'\"]+)",
    "phone": r"\+?\d{1,3}[\s-]?\d{3,4}[\s-]?\d{3,4}[\s-]?\d{0,4}",
    "person_name_context": r"(?:D\.|Mr\.|Mrs\.|Ms\.|Dr\.|Contacto:?)\s+[A-Z][a-záéíóúñ]+\s+[A-Z][a-záéíóúñ]+",
    "client_id": r"CLI-\d{3,}",
    "swift_code": r"\b[A-Z]{4}[A-Z]{2}[A-Z0-9]{2}([A-Z0-9]{3})?\b",
    "monetary_value": r"€\s?[\d.,]+(?:\s?(?:million|mil))",
}


def detect_pii_in_text(text: str) -> dict[str, list[str]]:
    """Detect PII matches in a text block. Returns dict of pattern_name -> matches."""
    findings: dict[str, list[str]] = {}
    for name, pattern in PII_PATTERNS.items():
        matches = re.findall(pattern, text)
        if matches:
            findings[name] = matches
    return findings


def compute_metrics(chunks: list[Document], strategy_name: str, doc_name: str) -> dict:
    """Compute quality metrics for a set of chunks."""
    sizes = [len(c.page_content) for c in chunks]
    
    # Boundary coherence: chunk ends on sentence-ending punctuation
    sentence_endings = sum(
        1 for c in chunks
        if c.page_content.rstrip()[-1:] in {".", "!", "?", ":"}
    )
    boundary_coherence = sentence_endings / len(chunks) if chunks else 0
    
    # PII isolation: what % of chunks with PII are "PII-only" (< 200 chars or marked)
    chunks_with_pii = [c for c in chunks if detect_pii_in_text(c.page_content)]
    total_pii_types = set()
    for c in chunks_with_pii:
        total_pii_types.update(detect_pii_in_text(c.page_content).keys())
    
    # PII spread: across how many chunks is PII distributed (lower = better isolation)
    pii_spread = len(chunks_with_pii)
    
    return {
        "strategy": strategy_name,
        "document": doc_name,
        "num_chunks": len(chunks),
        "avg_size": round(statistics.mean(sizes)) if sizes else 0,
        "min_size": min(sizes) if sizes else 0,
        "max_size": max(sizes) if sizes else 0,
        "std_size": round(statistics.stdev(sizes)) if len(sizes) > 1 else 0,
        "boundary_coherence": round(boundary_coherence, 3),
        "chunks_with_pii": pii_spread,
        "pii_types_found": sorted(total_pii_types),
    }


def print_chunks_detail(chunks: list[Document], max_display: int = 5):
    """Print chunk details for inspection."""
    for i, chunk in enumerate(chunks[:max_display]):
        pii = detect_pii_in_text(chunk.page_content)
        pii_flag = f" [PII: {list(pii.keys())}]" if pii else ""
        print(f"  Chunk {i}: {len(chunk.page_content)} chars{pii_flag}")
        preview = chunk.page_content[:120].replace('\n', ' ')
        print(f"    > {preview}...")
    if len(chunks) > max_display:
        print(f"  ... and {len(chunks) - max_display} more chunks")


# Store all metrics for final comparison
all_metrics: list[dict] = []
# Store all chunk results for notebook 03
all_chunk_results: dict[str, dict[str, list[Document]]] = {}

print("Metrics framework ready.")
print(f"PII patterns registered: {list(PII_PATTERNS.keys())}")

Metrics framework ready.
PII patterns registered: ['email', 'iban', 'ip_address', 'password', 'phone', 'person_name_context', 'client_id', 'swift_code', 'monetary_value']


---
## Strategy 1: Fixed Chunking (with overlap)

Uses `RecursiveCharacterTextSplitter` with a fixed `chunk_size` and `chunk_overlap`. This is the baseline strategy - simple and deterministic.

In [3]:
FIXED_CHUNK_SIZE = 500
FIXED_CHUNK_OVERLAP = 100

fixed_splitter = RecursiveCharacterTextSplitter(
    chunk_size=FIXED_CHUNK_SIZE,
    chunk_overlap=FIXED_CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)

fixed_results: dict[str, list[Document]] = {}

for filename, docs in all_documents.items():
    # Merge all sections into single text, preserving source metadata
    full_text = "\n\n".join(d.page_content for d in docs)
    base_metadata = docs[0].metadata.copy()
    
    chunks = fixed_splitter.create_documents(
        texts=[full_text],
        metadatas=[base_metadata],
    )
    
    # Add chunk_id
    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = f"fixed_{i:03d}"
    
    fixed_results[filename] = chunks
    metrics = compute_metrics(chunks, "fixed", filename)
    all_metrics.append(metrics)
    
    print(f"\n--- {filename} ---")
    print(f"Chunks: {metrics['num_chunks']} | Avg size: {metrics['avg_size']} | Boundary coherence: {metrics['boundary_coherence']}")
    print_chunks_detail(chunks, max_display=3)

all_chunk_results["fixed"] = fixed_results


--- Witty-QuickGuide-EN.pdf ---
Chunks: 7 | Avg size: 357 | Boundary coherence: 0.0
  Chunk 0: 444 chars [PII: ['phone']]
    > ENG QUICK GUIDE Witty Manager Software The USB stick contains the Witty Manager software, as well as the relevant user m...
  Chunk 1: 152 chars [PII: ['email', 'phone']]
    > www.microgate.it/witty Via Waltraud Gebert Deeg, 3e • Bolzano • Italy  Tel. +39 0471 501532  info@microgate.it  •  www.m...
  Chunk 2: 413 chars
    > Documentation The Witty Kit user manual is stored on the USB stick  inside the backpack pocket. Please open or print the...
  ... and 4 more chunks

--- Witty-Financial-Report-2025.pdf ---
Chunks: 6 | Avg size: 456 | Boundary coherence: 0.333
  Chunk 0: 480 chars [PII: ['swift_code']]
    > MICROGATE S.R.L. - QUARTERLY FINANCIAL REPORT (Q3 2025)  Department: Finance Department  Confidentiality Level: STRICTLY...
  Chunk 1: 492 chars [PII: ['monetary_value']]
    > Timer, was once again the primary revenue driver. Adoption of our wireless

---
## Strategy 2: Structural Chunking (logical separators)

Splits on logical/structural boundaries: numbered sections, headers, timestamps (for logs), and paragraph breaks. The key difference from fixed chunking is that boundaries are content-aware.

In [4]:
# Structural separators ordered by priority (most specific first)
STRUCTURAL_SEPARATORS = [
    r"\n(?=\d+\.\s+[A-Z])",          # Numbered sections: "1. Executive Summary"
    r"\n(?=PRIMERA|SEGUNDA|TERCERA|CL[ÁA]USULAS)",  # Contract clauses
    r"\n(?=\[\d{4}-)",               # Log timestamps: "[2026-03-05..."
    r"\n(?=Sheet:)",                  # Excel sheet boundaries
    r"\n(?=#{1,3}\s)",               # Markdown headers
    r"\n\n",                          # Double newline (paragraph)
]


def structural_split(text: str, min_chunk_size: int = 80) -> list[str]:
    """Split text on structural boundaries, merging small fragments."""
    # Try each separator pattern, use the first that produces meaningful splits
    best_chunks = [text]  # fallback: no split
    
    for pattern in STRUCTURAL_SEPARATORS:
        parts = re.split(pattern, text)
        parts = [p.strip() for p in parts if p.strip()]
        if len(parts) > 1:
            # Merge small fragments with the next chunk
            merged = []
            buffer = ""
            for part in parts:
                if buffer and len(buffer) >= min_chunk_size:
                    merged.append(buffer)
                    buffer = part
                else:
                    buffer = (buffer + "\n" + part).strip() if buffer else part
            if buffer:
                merged.append(buffer)
            
            if len(merged) > len(best_chunks):
                best_chunks = merged
    
    return best_chunks


structural_results: dict[str, list[Document]] = {}

for filename, docs in all_documents.items():
    full_text = "\n\n".join(d.page_content for d in docs)
    base_metadata = docs[0].metadata.copy()
    
    text_chunks = structural_split(full_text)
    chunks = []
    for i, text in enumerate(text_chunks):
        meta = base_metadata.copy()
        meta["chunk_id"] = f"struct_{i:03d}"
        chunks.append(Document(page_content=text, metadata=meta))
    
    structural_results[filename] = chunks
    metrics = compute_metrics(chunks, "structural", filename)
    all_metrics.append(metrics)
    
    print(f"\n--- {filename} ---")
    print(f"Chunks: {metrics['num_chunks']} | Avg size: {metrics['avg_size']} | Boundary coherence: {metrics['boundary_coherence']}")
    print_chunks_detail(chunks, max_display=3)

all_chunk_results["structural"] = structural_results


--- Witty-QuickGuide-EN.pdf ---
Chunks: 2 | Avg size: 1124 | Boundary coherence: 0.0
  Chunk 0: 505 chars [PII: ['email', 'phone']]
    > ENG QUICK GUIDE Witty Manager Software The USB stick contains the Witty Manager software, as well as the relevant user m...
  Chunk 1: 1743 chars
    > Documentation The Witty Kit user manual is stored on the USB stick  inside the backpack pocket. Please open or print the...

--- Witty-Financial-Report-2025.pdf ---
Chunks: 5 | Avg size: 488 | Boundary coherence: 0.8
  Chunk 0: 215 chars [PII: ['swift_code']]
    > MICROGATE S.R.L. - QUARTERLY FINANCIAL REPORT (Q3 2025)  Department: Finance Department  Confidentiality Level: STRICTLY...
  Chunk 1: 353 chars
    > 1. Executive Summary  The third quarter of 2025 solidified Microgate's position as a leader in sports  timing, based in ...
  Chunk 2: 851 chars [PII: ['monetary_value']]
    > 2. Revenue Breakdown by Product  Gross revenue for the sports division reached €4.2 million this quarter. Sales are

---
## Strategy 3: Semantic Chunking (embedding similarity)

Splits text into sentences, computes embeddings, and groups consecutive sentences that are semantically similar. Uses a local HuggingFace model (no external API calls - suitable for on-premise).

In [5]:
# Load embedding model (local, on-premise compatible)
print("Loading embedding model (first run downloads ~90MB)...")
embeddings_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)
print("Embedding model ready.")

Loading embedding model (first run downloads ~90MB)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

D:\URV\TFG\ai-rag-context-auth-system\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\arnau\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model ready.


In [6]:
def split_into_sentences(text: str) -> list[str]:
    """Split text into sentences using regex. Handles common abbreviations."""
    # Split on sentence-ending punctuation followed by space+uppercase or newline
    sentences = re.split(r'(?<=[.!?:])\s+(?=[A-Z\[€(])', text)
    # Also split on newlines for log-style content
    result = []
    for s in sentences:
        sub = s.split("\n")
        result.extend([x.strip() for x in sub if x.strip()])
    return result


def semantic_chunk(
    text: str,
    similarity_threshold: float = 0.5,
    min_chunk_sentences: int = 2,
) -> list[str]:
    """Group consecutive sentences by embedding cosine similarity."""
    sentences = split_into_sentences(text)
    if len(sentences) <= min_chunk_sentences:
        return [text]
    
    # Compute embeddings for all sentences
    sentence_embeddings = embeddings_model.embed_documents(sentences)
    embeddings_array = np.array(sentence_embeddings)
    
    # Compute cosine similarity between consecutive sentences
    similarities = []
    for i in range(len(embeddings_array) - 1):
        a, b = embeddings_array[i], embeddings_array[i + 1]
        cos_sim = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)
        similarities.append(cos_sim)
    
    # Group sentences: split when similarity drops below threshold
    groups: list[list[str]] = [[sentences[0]]]
    for i, sim in enumerate(similarities):
        if sim < similarity_threshold:
            groups.append([sentences[i + 1]])
        else:
            groups[-1].append(sentences[i + 1])
    
    # Merge groups that are too small
    merged = []
    buffer: list[str] = []
    for group in groups:
        buffer.extend(group)
        if len(buffer) >= min_chunk_sentences:
            merged.append(" ".join(buffer))
            buffer = []
    if buffer:
        if merged:
            merged[-1] += " " + " ".join(buffer)
        else:
            merged.append(" ".join(buffer))
    
    return merged


print("Semantic chunking function ready.")

Semantic chunking function ready.


In [7]:
SEMANTIC_THRESHOLD = 0.45

semantic_results: dict[str, list[Document]] = {}

for filename, docs in all_documents.items():
    full_text = "\n\n".join(d.page_content for d in docs)
    base_metadata = docs[0].metadata.copy()
    
    text_chunks = semantic_chunk(full_text, similarity_threshold=SEMANTIC_THRESHOLD)
    chunks = []
    for i, text in enumerate(text_chunks):
        meta = base_metadata.copy()
        meta["chunk_id"] = f"semantic_{i:03d}"
        chunks.append(Document(page_content=text, metadata=meta))
    
    semantic_results[filename] = chunks
    metrics = compute_metrics(chunks, "semantic", filename)
    all_metrics.append(metrics)
    
    print(f"\n--- {filename} ---")
    print(f"Chunks: {metrics['num_chunks']} | Avg size: {metrics['avg_size']} | Boundary coherence: {metrics['boundary_coherence']}")
    print_chunks_detail(chunks, max_display=3)

all_chunk_results["semantic"] = semantic_results


--- Witty-QuickGuide-EN.pdf ---
Chunks: 25 | Avg size: 88 | Boundary coherence: 0.28
  Chunk 0: 15 chars
    > ENG QUICK GUIDE...
  Chunk 1: 142 chars
    > Witty Manager Software The USB stick contains the Witty Manager software, as well as the relevant user manual, which you...
  Chunk 2: 192 chars
    > To execute the program, the PC must run Windows OS (XP/Vista/7/8). The steps for installing (chapter 2) and using the so...
  ... and 22 more chunks



--- Witty-Financial-Report-2025.pdf ---
Chunks: 29 | Avg size: 82 | Boundary coherence: 0.517
  Chunk 0: 86 chars
    > MICROGATE S.R.L. - QUARTERLY FINANCIAL REPORT (Q3 2025) Department: Finance Department...
  Chunk 1: 62 chars [PII: ['swift_code']]
    > Confidentiality Level: STRICTLY CONFIDENTIAL (Management Only)...
  Chunk 2: 62 chars
    > Subject: Witty Product Line Performance and Year-End Forecasts...
  ... and 26 more chunks

--- distribution-contract-2026.docx ---
Chunks: 8 | Avg size: 132 | Boundary coherence: 1.0
  Chunk 0: 336 chars [PII: ['person_name_context', 'swift_code']]
    > CONTRATO DE DISTRIBUCIÓN EXCLUSIVA - MICROGATE S.R.L. REUNIDOS De una parte, Microgate S.R.L., con domicilio en Via Walt...
  Chunk 1: 18 chars
    > CLÁUSULAS PRIMERA:...
  Chunk 2: 215 chars
    > Objeto del Contrato EL FABRICANTE otorga a EL DISTRIBUIDOR los derechos exclusivos para la comercialización del producto...
  ... and 5 more chunks

--- server_logs_witty_backend.txt ---
Chunks:


--- clients-and-billings.xlsx ---
Chunks: 3 | Avg size: 175 | Boundary coherence: 0.0
  Chunk 0: 13 chars
    > Sheet: Sheet1...
  Chunk 1: 206 chars [PII: ['email', 'person_name_context', 'client_id']]
    > ID_Cliente                    Empresa     Contacto                  Email  Facturación_YTD Nivel_Soporte CLI-001 Centro ...
  Chunk 2: 305 chars [PII: ['email', 'client_id']]
    > CLI-002    Federación de Atletismo  Marcos Ruiz mruiz@fedatletismo.org             4200      Standard CLI-003    Univers...


---
## Strategy 4: Custom RBAC-Aware Chunking

**This is the core innovation for the TFG.** The custom strategy:

1. First performs structural splitting (content-aware boundaries)
2. Scans each chunk for sensitive content (PII, credentials, financial data)
3. **Isolates** sensitive fragments into dedicated small chunks
4. Marks isolated chunks with elevated `clearance_level` metadata

This ensures that even in a globally "public" document, a chunk containing an exposed password or IBAN gets its own elevated clearance - preventing data leaks through the retrieval pipeline.

In [8]:
# Sensitive content patterns with their clearance escalation level
SENSITIVE_PATTERNS: list[dict] = [
    {"name": "password", "pattern": r"(?i)(?:password|contrase[ñn]a|pwd|override[_ ]?password)\s*[:=]\s*['\"]?[^\s'\"]+['\"]?", "escalate_to": 3},
    {"name": "iban", "pattern": r"(?:IBAN|Cuenta)[:\s]*[A-Z]{2}\d{2}[\s]?[A-Z0-9\s]{15,30}", "escalate_to": 3},
    {"name": "swift", "pattern": r"SWIFT[:\s]*[A-Z]{4}[A-Z]{2}[A-Z0-9]{2,5}", "escalate_to": 3},
    {"name": "email_personal", "pattern": r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", "escalate_to": 2},
    {"name": "ip_address", "pattern": r"\b(?:from IP|IP)\s*\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b", "escalate_to": 2},
    {"name": "client_id", "pattern": r"CLI-\d{3,}", "escalate_to": 2},
    {"name": "person_name", "pattern": r"(?:D\.|Mr\.|Mrs\.|Contacto:?)\s+[A-Z][a-záéíóúñ]+\s+[A-Z][a-záéíóúñ]+", "escalate_to": 2},
    {"name": "monetary_confidential", "pattern": r"(?:coste|cost|precio|price|margen|margin)[^.]*€\s?[\d.,]+", "escalate_to": 2},
]


def find_sensitive_spans(text: str) -> list[dict]:
    """Find all sensitive content spans in text with their escalation level."""
    spans = []
    for sp in SENSITIVE_PATTERNS:
        for match in re.finditer(sp["pattern"], text):
            # Expand to include surrounding context (the full line)
            start = text.rfind("\n", 0, match.start())
            start = start + 1 if start != -1 else match.start()
            end = text.find("\n", match.end())
            end = end if end != -1 else match.end()
            spans.append({
                "type": sp["name"],
                "start": start,
                "end": end,
                "escalate_to": sp["escalate_to"],
                "text": text[start:end],
            })
    # Sort by start position and merge overlapping spans
    spans.sort(key=lambda x: x["start"])
    return spans


def custom_rbac_chunk(text: str, base_metadata: dict, min_chunk_size: int = 80) -> list[Document]:
    """Custom RBAC-aware chunking that isolates sensitive content."""
    # Step 1: Structural split first
    structural_parts = structural_split(text, min_chunk_size=min_chunk_size)
    
    result_chunks: list[Document] = []
    
    for part in structural_parts:
        sensitive_spans = find_sensitive_spans(part)
        
        if not sensitive_spans:
            # No sensitive content - keep as-is
            meta = base_metadata.copy()
            meta["contains_PII"] = False
            meta["sensitivity_types"] = []
            result_chunks.append(Document(page_content=part, metadata=meta))
        else:
            # Isolate sensitive content into dedicated chunks
            used_ranges: list[tuple[int, int]] = []
            
            for span in sensitive_spans:
                # Create isolated chunk for sensitive content
                meta = base_metadata.copy()
                meta["contains_PII"] = True
                meta["sensitivity_types"] = [span["type"]]
                # Escalate clearance if the detected sensitivity exceeds document level
                meta["clearance_level"] = max(meta.get("clearance_level", 0), span["escalate_to"])
                result_chunks.append(Document(page_content=span["text"], metadata=meta))
                used_ranges.append((span["start"], span["end"]))
            
            # Create chunk(s) for the non-sensitive remainder
            # Sort and merge overlapping used ranges
            used_ranges.sort()
            merged_ranges: list[tuple[int, int]] = []
            for start, end in used_ranges:
                if merged_ranges and start <= merged_ranges[-1][1]:
                    merged_ranges[-1] = (merged_ranges[-1][0], max(merged_ranges[-1][1], end))
                else:
                    merged_ranges.append((start, end))
            
            # Extract non-sensitive text between used ranges
            prev_end = 0
            remainder_parts = []
            for start, end in merged_ranges:
                if start > prev_end:
                    fragment = part[prev_end:start].strip()
                    if fragment:
                        remainder_parts.append(fragment)
                prev_end = end
            # Trailing text
            if prev_end < len(part):
                fragment = part[prev_end:].strip()
                if fragment:
                    remainder_parts.append(fragment)
            
            if remainder_parts:
                remainder_text = "\n".join(remainder_parts)
                meta = base_metadata.copy()
                meta["contains_PII"] = False
                meta["sensitivity_types"] = []
                result_chunks.append(Document(page_content=remainder_text, metadata=meta))
    
    # Assign chunk IDs
    for i, chunk in enumerate(result_chunks):
        chunk.metadata["chunk_id"] = f"custom_{i:03d}"
    
    return result_chunks


print(f"Custom RBAC strategy ready. Sensitive patterns: {len(SENSITIVE_PATTERNS)}")
for sp in SENSITIVE_PATTERNS:
    print(f"  {sp['name']} -> escalates to clearance {sp['escalate_to']}")

Custom RBAC strategy ready. Sensitive patterns: 8
  password -> escalates to clearance 3
  iban -> escalates to clearance 3
  swift -> escalates to clearance 3
  email_personal -> escalates to clearance 2
  ip_address -> escalates to clearance 2
  client_id -> escalates to clearance 2
  person_name -> escalates to clearance 2
  monetary_confidential -> escalates to clearance 2


In [9]:
custom_results: dict[str, list[Document]] = {}

for filename, docs in all_documents.items():
    full_text = "\n\n".join(d.page_content for d in docs)
    base_metadata = docs[0].metadata.copy()
    
    chunks = custom_rbac_chunk(full_text, base_metadata)
    
    custom_results[filename] = chunks
    metrics = compute_metrics(chunks, "custom_rbac", filename)
    all_metrics.append(metrics)
    
    print(f"\n--- {filename} ---")
    print(f"Chunks: {metrics['num_chunks']} | Avg size: {metrics['avg_size']} | Boundary coherence: {metrics['boundary_coherence']}")
    print(f"PII chunks: {metrics['chunks_with_pii']} | PII types: {metrics['pii_types_found']}")
    
    # Show PII isolation detail
    for i, chunk in enumerate(chunks):
        if chunk.metadata.get("contains_PII"):
            print(f"  [ISOLATED] Chunk {i}: clearance={chunk.metadata['clearance_level']}, "
                  f"types={chunk.metadata['sensitivity_types']}")
            print(f"    > {chunk.page_content[:100]}...")

all_chunk_results["custom_rbac"] = custom_results


--- Witty-QuickGuide-EN.pdf ---
Chunks: 3 | Avg size: 748 | Boundary coherence: 0.0
PII chunks: 2 | PII types: ['email', 'phone']
  [ISOLATED] Chunk 0: clearance=2, types=['email_personal']
    > info@microgate.it...

--- Witty-Financial-Report-2025.pdf ---
Chunks: 5 | Avg size: 488 | Boundary coherence: 0.8
PII chunks: 3 | PII types: ['monetary_value', 'swift_code']

--- distribution-contract-2026.docx ---
Chunks: 7 | Avg size: 153 | Boundary coherence: 0.571
PII chunks: 4 | PII types: ['iban', 'person_name_context', 'phone', 'swift_code']
  [ISOLATED] Chunk 0: clearance=2, types=['person_name']
    > REUNIDOS De una parte, Microgate S.R.L., con domicilio en Via Waltraud Gebert Deeg, 3e Bolzano, Ital...
  [ISOLATED] Chunk 4: clearance=3, types=['iban']
    > IBAN: IT89 A012 3456 7890 1234 5678 901 SWIFT: MCRGIT2M...
  [ISOLATED] Chunk 5: clearance=3, types=['swift']
    > SWIFT: MCRGIT2M...

--- server_logs_witty_backend.txt ---
Chunks: 8 | Avg size: 81 | Boundary coherence: 0.625
PI

---
## Comparative Summary Table

All metrics side-by-side for Phase 2 analysis.

In [10]:
import pandas as pd

df_metrics = pd.DataFrame(all_metrics)

# Display per-document comparison
print("=" * 90)
print("FULL METRICS TABLE")
print("=" * 90)
display_cols = ["strategy", "document", "num_chunks", "avg_size", "min_size", "max_size", 
                "std_size", "boundary_coherence", "chunks_with_pii"]
print(df_metrics[display_cols].to_string(index=False))

# Aggregated by strategy
print("\n" + "=" * 90)
print("AGGREGATED BY STRATEGY")
print("=" * 90)
agg = df_metrics.groupby("strategy").agg({
    "num_chunks": ["sum", "mean"],
    "avg_size": "mean",
    "boundary_coherence": "mean",
    "chunks_with_pii": "sum",
}).round(3)
print(agg)

FULL METRICS TABLE
   strategy                        document  num_chunks  avg_size  min_size  max_size  std_size  boundary_coherence  chunks_with_pii
      fixed         Witty-QuickGuide-EN.pdf           7       357       115       468       154               0.000                2
      fixed Witty-Financial-Report-2025.pdf           6       456       396       492        38               0.333                3
      fixed distribution-contract-2026.docx           3       356       310       411        51               0.667                2
      fixed   server_logs_witty_backend.txt           2       374       286       462       124               1.000                2
      fixed       clients-and-billings.xlsx           2       267       101       433       235               0.000                2
 structural         Witty-QuickGuide-EN.pdf           2      1124       505      1743       875               0.000                1
 structural Witty-Financial-Report-2025.pdf       

In [11]:
# Highlight: PII isolation effectiveness per strategy
print("\n" + "=" * 90)
print("PII ISOLATION ANALYSIS")
print("=" * 90)

# For each strategy, check how many chunks contain PII and how concentrated it is
for strategy_name, results in all_chunk_results.items():
    print(f"\n--- {strategy_name.upper()} ---")
    total_chunks = 0
    total_pii_chunks = 0
    pii_only_chunks = 0  # Chunks where PII is isolated (small, dedicated)
    
    for filename, chunks in results.items():
        for chunk in chunks:
            total_chunks += 1
            pii = detect_pii_in_text(chunk.page_content)
            if pii:
                total_pii_chunks += 1
                # A "PII-only" chunk is one where sensitive content is the majority
                if chunk.metadata.get("contains_PII") or len(chunk.page_content) < 200:
                    pii_only_chunks += 1
    
    pii_concentration = pii_only_chunks / total_pii_chunks if total_pii_chunks > 0 else 0
    print(f"  Total chunks: {total_chunks}")
    print(f"  Chunks with PII: {total_pii_chunks}")
    print(f"  PII isolated (dedicated chunks): {pii_only_chunks}")
    print(f"  PII concentration ratio: {pii_concentration:.1%}")


PII ISOLATION ANALYSIS

--- FIXED ---
  Total chunks: 20
  Chunks with PII: 11
  PII isolated (dedicated chunks): 2
  PII concentration ratio: 18.2%

--- STRUCTURAL ---
  Total chunks: 18
  Chunks with PII: 11
  PII isolated (dedicated chunks): 4
  PII concentration ratio: 36.4%

--- SEMANTIC ---
  Total chunks: 74
  Chunks with PII: 14
  PII isolated (dedicated chunks): 11
  PII concentration ratio: 78.6%

--- CUSTOM_RBAC ---
  Total chunks: 33
  Chunks with PII: 22
  PII isolated (dedicated chunks): 18
  PII concentration ratio: 81.8%


## Export Results for Notebook 03

In [ ]:
# Export metrics
df_metrics.to_csv("../../data/results/notebook_results/chunking_metrics.csv", index=False)
print("Metrics exported to ../../data/results/notebook_results/chunking_metrics.csv")

# Export chunk results for notebook 03
serialized_chunks = {}
for strategy, results in all_chunk_results.items():
    serialized_chunks[strategy] = {}
    for filename, chunks in results.items():
        serialized_chunks[strategy][filename] = [
            {"page_content": c.page_content, "metadata": c.metadata}
            for c in chunks
        ]

with open("../../../data/results/notebook_results/chunk_results.json", "w", encoding="utf-8") as f:
    json.dump(serialized_chunks, f, ensure_ascii=False, indent=2)

print("Chunk results exported to ../../data/results/notebook_results/chunk_results.json")
print(f"\nStrategies exported: {list(serialized_chunks.keys())}")
for strategy in serialized_chunks:
    total = sum(len(v) for v in serialized_chunks[strategy].values())
    print(f"  {strategy}: {total} total chunks")